# 神经网络结构与训练框架


### 🧩 第1节：nn.Module 与多层感知机（MLP）结构
🎯 学习目标

通过这一节，你将掌握：

1. torch.nn.Module 的基本用法

2. 如何定义神经网络层（线性层 + 激活函数）

3. 前向传播 forward() 的机制

4. 模型对象的使用与参数访问

#### 最小可运行的 MLP 模型

In [3]:
import torch
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP()
print(model)

x = torch.tensor([[1.0, 2.0]])
y = model(x)
print(y)
        

MLP(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=4, out_features=1, bias=True)
)
tensor([[0.3239]], grad_fn=<AddmmBackward0>)


非常好 👍！
现在你已经能成功定义并运行一个最基础的多层感知机（MLP）模型了。
接下来我们进入 下一步学习阶段 —— 在这个阶段，我们将让模型学会拟合数据，也就是让它通过训练自动调整参数。

#### 训练一个简单的 MLP
目标：让模型学会拟合一个二维输入到标量输出的函数

我们手动生成一些训练数据，例如：

In [ ]:
# 1. 准备数据

import torch

x = torch.rand(100, 2)                  # (100, 2)
y = 3 * x[:, 0] + 2 * x[:, 1] + 1       # (100, )
y = y.unsqueeze(1)                      # (100, 1)


# 2. 定义模型

import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = MLP()


# 3. 定义损失函数和优化器

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)


# 4. 训练模型

for epoch in range(200):
    # 前向传播
    y_pred = model(x)
    loss = criterion(y_pred, y)
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.6f}")
        

# 5. 查看效果

print("训练后的预测")
test = torch.tensor([[0.5, 0.5]])
print("模型预测结果：", model(test).item())
print("真实结果", 3 * 0.5 + 2 * 0.5 + 1)

Epoch   0 | Loss: 14.097671
Epoch  20 | Loss: 0.168003
Epoch  40 | Loss: 0.017898
Epoch  60 | Loss: 0.001132
Epoch  80 | Loss: 0.000076
Epoch 100 | Loss: 0.000021
Epoch 120 | Loss: 0.000018
Epoch 140 | Loss: 0.000018
Epoch 160 | Loss: 0.000018
Epoch 180 | Loss: 0.000017
训练后的预测
模型预测结果： 3.5011627674102783
真实结果 3.5


### 🧩 第2节：在 MNIST 数据集上实践

#### LeNet

In [1]:
# 一、导入依赖和数据集

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用设备：", device)

# 1. 数据预处理
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307, ), (0.3081, ))
])

# 2. 下载训练集和测试集
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform)

# 3. 使用dataloader按批加载
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


# 二、定义LeNet网络结构

class LeNet(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = LeNet().to(device)
print(model)


# 三、定义损失函数和优化器

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)


# 四、训练

for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        outputs = model(data)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
    print(f"Epoch {epoch} 平均Loss: {running_loss / len(train_loader):.4f}")
    
    
# 五、在测试集上评估

model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

print(f"在测试集上准确率：{100 * correct / total:.2f}%")

使用设备： cuda
LeNet(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=256, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)
Epoch [1], Step [0/938], Loss: 2.2937
Epoch [1], Step [100/938], Loss: 0.1862
Epoch [1], Step [200/938], Loss: 0.1807
Epoch [1], Step [300/938], Loss: 0.3618
Epoch [1], Step [400/938], Loss: 0.1056
Epoch [1], Step [500/938], Loss: 0.0414
Epoch [1], Step [600/938], Loss: 0.2296
Epoch [1], Step [700/938], Loss: 0.1283
Epoch [1], Step [800/938], Loss: 0.1252
Epoch [1], Step [900/938], Loss: 0.1786
Epoch 1 平均Loss: 0.2730
Epoch [2], Step [0/938], Loss: 0.1015
Epoch [2], Step [100/938], Loss: 0.0042
Epoch [2], Step [200/938], Loss: 0.0222
Epoch [2], Step [300/938], Loss: 0.1583
Epoch [2], Step [400/938], Loss: 0.0295
Epo